In [1]:
!pip install line_profiler

In [2]:
%load_ext line_profiler

In [3]:
import random

def create_grid(rows, cols, alive):
    grid = [[0 for x in range(cols)] for y in range(rows)]
    count = int(rows * cols * alive)
    while count:
        row, col = random.randrange(rows), random.randrange(cols)
        if grid[row][col] == 0:
            grid[row][col] = 1
            count -= 1
    return grid

In [4]:
%lprun -f create_grid grid = create_grid(500, 500, 0.5)

Timer unit: 1e-07 s

Total time: 1.9009 s
File: C:\Users\Serkan\AppData\Local\Temp\ipykernel_1780\239733883.py
Function: create_grid at line 3

Line #      Hits         Time  Per Hit   % Time  Line Contents
     3                                           def create_grid(rows, cols, alive):
     4         1      77484.0  77484.0      0.4      grid = [[0 for x in range(cols)] for y in range(rows)]
     5         1         41.0     41.0      0.0      count = int(rows * cols * alive)
     6    173058     703707.0      4.1      3.7      while count:
     7    173057   16300578.0     94.2     85.8          row, col = random.randrange(rows), random.randrange(cols)
     8    173057     907079.0      5.2      4.8          if grid[row][col] == 0:
     9    125000     511684.0      4.1      2.7              grid[row][col] = 1
    10    125000     508423.0      4.1      2.7              count -= 1
    11         1         19.0     19.0      0.0      return grid

In [5]:
%%timeit -n 5 -r 3
grid = create_grid(500, 500, 0.5)

170 ms ± 275 μs per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [6]:
def save_grid(grid, filename):
    with open(filename, "w") as file:
        rows, cols = len(grid), len(grid[0])
        file.write(f"{rows},{cols}\n")
        for y, row in enumerate(grid):
            for x, cell in enumerate(row):
                if cell:
                    file.write(f"{y},{x}\n")

In [7]:
%lprun -f save_grid save_grid(grid, "input_500x500_0.5.txt")

Timer unit: 1e-07 s

Total time: 0.430576 s
File: C:\Users\Serkan\AppData\Local\Temp\ipykernel_1780\1827802528.py
Function: save_grid at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def save_grid(grid, filename):
     2         2       4085.0   2042.5      0.1      with open(filename, "w") as file:
     3         1         19.0     19.0      0.0          rows, cols = len(grid), len(grid[0])
     4         1         90.0     90.0      0.0          file.write(f"{rows},{cols}\n")
     5       501       2865.0      5.7      0.1          for y, row in enumerate(grid):
     6    250500    1093378.0      4.4     25.4              for x, cell in enumerate(row):
     7    250000     953617.0      3.8     22.1                  if cell:
     8    125000    2251706.0     18.0     52.3                      file.write(f"{y},{x}\n")

In [8]:
%%timeit -n 5 -r 3
save_grid(grid, "input_500x500_0.5.txt")

85.2 ms ± 213 μs per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [9]:
def read_grid(filename):
    with open(filename) as file:
        rows, cols = map(int, file.readline().split(','))
        grid = [[0 for x in range(cols)] for y in range(rows)]
        for line in file:
            row, col = map(int, line.split(','))
            grid[row][col] = 1
    return grid

In [10]:
%lprun -f read_grid grid = read_grid("input_500x500_0.5.txt")

Timer unit: 1e-07 s

Total time: 0.240145 s
File: C:\Users\Serkan\AppData\Local\Temp\ipykernel_1780\4001913799.py
Function: read_grid at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def read_grid(filename):
     2         2      94607.0  47303.5      3.9      with open(filename) as file:
     3         1        682.0    682.0      0.0          rows, cols = map(int, file.readline().split(','))
     4         1      94059.0  94059.0      3.9          grid = [[0 for x in range(cols)] for y in range(rows)]
     5    125001     628093.0      5.0     26.2          for line in file:
     6    125000    1006129.0      8.0     41.9              row, col = map(int, line.split(','))
     7    125000     577867.0      4.6     24.1              grid[row][col] = 1
     8         1         14.0     14.0      0.0      return grid

In [11]:
%%timeit -n 5 -r 3
grid = read_grid("input_500x500_0.5.txt")

60 ms ± 448 μs per loop (mean ± std. dev. of 3 runs, 5 loops each)


- Any **live cell with fewer than two live neighbours dies**, as if by underpopulation.
- Any **live cell with two or three live neighbours lives** on to the next generation.
- Any **live cell with more than three live neighbours dies**, as if by overpopulation.
- Any **dead cell with exactly three live neighbours becomes a live cell**, as if by reproduction.

In [12]:
def tick(grid):
    rows = len(grid)
    cols = len(grid[0])
    new_grid = [[0 for x in range(cols)] for y in range(rows)]
    for row in range(rows):
        for col in range(cols):
            state = grid[row][col]
            count = 0
            for d_row in range(-1, 2):
                for d_col in range(-1, 2):
                    n_row, n_col = row + d_row, col + d_col
                    if n_row >= 0 and n_row < rows and n_col >= 0 and n_col < cols:
                        if n_row != row or  n_col != col:
                            n_state = grid[n_row][n_col]
                            if n_state == 1:
                                count += 1
            if state == 1:
                if count < 2:
                    new_state = 0
                elif count == 2 or count == 3:
                    new_state = 1
                elif count > 3:
                    new_state = 0
            else:
                if count == 3:
                    new_state = 1
                else:
                    new_state = 0
            
            new_grid[row][col] = new_state
    return new_grid

In [13]:
%lprun -f tick next_grid = tick(grid)

Timer unit: 1e-07 s

Total time: 7.5078 s
File: C:\Users\Serkan\AppData\Local\Temp\ipykernel_1780\212703557.py
Function: tick at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def tick(grid):
     2         1         13.0     13.0      0.0      rows = len(grid)
     3         1         11.0     11.0      0.0      cols = len(grid[0])
     4         1      78945.0  78945.0      0.1      new_grid = [[0 for x in range(cols)] for y in range(rows)]
     5       501       2058.0      4.1      0.0      for row in range(rows):
     6    250500    1033008.0      4.1      1.4          for col in range(cols):
     7    250000    1010426.0      4.0      1.3              state = grid[row][col]
     8    250000     970021.0      3.9      1.3              count = 0
     9   1000000    4230449.0      4.2      5.6              for d_row in range(-1, 2):
    10   3000000   14066012.0      4.7     18.7                  for d_col in r

In [14]:
%%timeit -n 5 -r 3
next_grid = tick(grid)

731 ms ± 6.1 ms per loop (mean ± std. dev. of 3 runs, 5 loops each)


In [15]:
grid = read_grid("input_5x5.txt")
next_grid = tick(grid)
next_grid

[[0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 1, 1, 0],
 [0, 1, 1, 1, 0],
 [0, 0, 0, 0, 0]]

In [16]:
next_grid = tick(next_grid)
next_grid

[[0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 1, 0],
 [0, 1, 0, 1, 0],
 [0, 0, 1, 0, 0]]